In [1]:
import pandas as pd

data_path = "../data/diabetes.csv"  # up one level, then into data/
df = pd.read_csv(data_path)

df.head()       # check first 5 rows
df.info()       # structure
df.describe()   # basic stats
df.isna().sum() # missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [2]:
# 1) Features (X) and target (y)
target_col = "Outcome"

X = df.drop(columns=[target_col])
y = df[target_col]

X.shape, y.shape

# 2) Train / val / test split
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

X_train.shape, X_val.shape, X_test.shape

((537, 8), (115, 8), (116, 8))

In [3]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit on training data ONLY (learn mean and std)
scaler.fit(X_train)

# Transform train, val, and test using the same scaler
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Quick check of shapes
X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape

((537, 8), (115, 8), (116, 8))

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Create the model
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Train (fit) the model on scaled training data
log_reg.fit(X_train_scaled, y_train)

# Predict on validation set
y_val_pred = log_reg.predict(X_val_scaled)

# Evaluate
val_accuracy = accuracy_score(y_val, y_val_pred)
print("Validation accuracy:", val_accuracy)

print("\nClassification report (validation):")
print(classification_report(y_val, y_val_pred))

Validation accuracy: 0.7304347826086957

Classification report (validation):
              precision    recall  f1-score   support

           0       0.78      0.83      0.80        75
           1       0.63      0.55      0.59        40

    accuracy                           0.73       115
   macro avg       0.70      0.69      0.69       115
weighted avg       0.72      0.73      0.73       115



In [5]:
import joblib
import os

# Make sure the models directory exists (relative to the notebook)
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

# Paths for the files
scaler_path = os.path.join(models_dir, "scaler.pkl")
log_reg_path = os.path.join(models_dir, "logistic_regression.pkl")

# Save the fitted scaler and model
joblib.dump(scaler, scaler_path)
joblib.dump(log_reg, log_reg_path)

['../models\\logistic_regression.pkl']

In [6]:
loaded_scaler = joblib.load(scaler_path)
loaded_log_reg = joblib.load(log_reg_path)

# Quick test: predict on validation set again
y_val_pred_loaded = loaded_log_reg.predict(loaded_scaler.transform(X_val))

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create the model (basic settings to start)
rf_clf = RandomForestClassifier(
    n_estimators=200,       # number of trees
    random_state=42,
    n_jobs=-1,             # use all CPU cores
)

# Train on the original (unscaled) features
rf_clf.fit(X_train, y_train)

# Predict on validation set
y_val_pred_rf = rf_clf.predict(X_val)

# Evaluate
rf_val_accuracy = accuracy_score(y_val, y_val_pred_rf)
print("Random Forest - Validation accuracy:", rf_val_accuracy)

print("\nRandom Forest - Classification report (validation):")
print(classification_report(y_val, y_val_pred_rf))

Random Forest - Validation accuracy: 0.7565217391304347

Random Forest - Classification report (validation):
              precision    recall  f1-score   support

           0       0.79      0.85      0.82        75
           1       0.68      0.57      0.62        40

    accuracy                           0.76       115
   macro avg       0.73      0.71      0.72       115
weighted avg       0.75      0.76      0.75       115



In [8]:
import joblib
import os

models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

rf_path = os.path.join(models_dir, "random_forest.pkl")

joblib.dump(rf_clf, rf_path)
print("Random Forest saved to:", rf_path)

Random Forest saved to: ../models\random_forest.pkl


In [3]:
import sys
print(sys.executable)

C:\Users\odwam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print("TF OK")

TF OK


In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X_train_scaled.shape[1]  # should be 8

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),      # 8 inputs
    layers.Dense(16, activation="relu"),   # hidden layer 1
    layers.Dense(8, activation="relu"),    # hidden layer 2
    layers.Dense(1, activation="sigmoid"), # output: probability (0-1)
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 289 (1.13 KB)

 Trainable params: 289 (1.13 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [12]:
batch_size = 32
epochs = 50

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
)

Epoch 1/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.6555 - loss: 0.6686 - val_accuracy: 0.6348 - val_loss: 0.6645
Epoch 2/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7002 - loss: 0.6500 - val_accuracy: 0.6609 - val_loss: 0.6470
Epoch 3/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7263 - loss: 0.6323 - val_accuracy: 0.6696 - val_loss: 0.6278
Epoch 4/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7393 - loss: 0.6126 - val_accuracy: 0.6870 - val_loss: 0.6091
Epoch 5/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7523 - loss: 0.5940 - val_accuracy: 0.6957 - val_loss: 0.5914
Epoch 6/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7579 - loss: 0.5751 - val_accuracy: 0.7043 - val_loss: 0.5747
Epoch 7/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7654 - loss: 0.5562 - val_accuracy: 0.7043 - val_loss: 0.5619
Epoch 8/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7654 - loss: 0.5399 - val_accuracy: 0.7043 - v

In [13]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print("Test accuracy:", test_accuracy)

Test accuracy: 0.7586206793785095


In [14]:
import os
from tensorflow import keras

models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

nn_path = os.path.join(models_dir, "diabetes_nn.keras")
model.save(nn_path)

print("Neural network saved to:", nn_path)

Neural network saved to: ../models\diabetes_nn.keras


In [4]:
# Compute and display saved-model accuracies for LR, RF, NN
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model

# Recreate the same data splits used earlier
df = pd.read_csv("../data/diabetes.csv")
X = df.drop(columns=["Outcome"])
y = df["Outcome"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

models_dir = "../models"
scaler = joblib.load(os.path.join(models_dir, "scaler.pkl"))
log_reg = joblib.load(os.path.join(models_dir, "logistic_regression.pkl"))
rf = joblib.load(os.path.join(models_dir, "random_forest.pkl"))
nn = load_model(os.path.join(models_dir, "diabetes_nn.keras"))

# Scale validation/test sets the same way as during training
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

val_acc_lr = accuracy_score(y_val, log_reg.predict(X_val_scaled))
val_acc_rf = accuracy_score(y_val, rf.predict(X_val))
test_loss_nn, test_acc_nn = nn.evaluate(X_test_scaled, y_test, verbose=0)

print("Logistic Regression - Validation accuracy:", round(val_acc_lr, 4))
print("Random Forest      - Validation accuracy:", round(val_acc_rf, 4))
print("Neural Network     - Test accuracy:      ", round(test_acc_nn, 4))

Logistic Regression - Validation accuracy: 0.7304
Random Forest      - Validation accuracy: 0.7565
Neural Network     - Test accuracy:       0.7586


# Minimal Evaluation notes

- Logistic Regression
    - Validation accuracy ≈ 0.73
- Random Forest
    - Validation accuracy ≈ 0.76
- Neural Network
    - Test accuracy ≈ 0.76

- Random Forest performed slightly better than Logistic Regression.
- Neural Network reached similar performance but is more complex to train and deploy.
- For this project, Random Forest was chosen as the main model because it is:
    - Easier to deploy, and
    - Achieves the best validation accuracy.